# Bi-encoder Cross-View Scoring

Encode RIENG CV-view va JD-view roi tinh cosine -> 50 dac trung tuong dong -> fusion.
**Run-All** se tu dong sweep 4 encoder x 4 fusion, luu checkpoint + plot + CSV ket qua.

In [1]:
# Cai dat thu vien (Kaggle/Colab)
!pip install -q transformers sentencepiece einops

In [ ]:
import os, json, gc, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             accuracy_score, precision_score, recall_score, f1_score)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
import warnings; warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [3]:
# =============================== CAU HINH ===============================
# Dataset da split (upload merged_dataset.zip lam Kaggle dataset slug: merged-cv-jd-scoring)
DATASET_BASE = "/kaggle/input/datasets/vuhuynguyen/en-vi-dataset-707-sample/merged_dataset"
CV_DIR     = os.path.join(DATASET_BASE, "cv")
JD_DIR     = os.path.join(DATASET_BASE, "jd")
SPLIT_ROOT = os.path.join(DATASET_BASE, "dataset")

WORK      = "/kaggle/working"
CACHE_DIR = os.path.join(WORK, "cache")
CKPT_DIR  = os.path.join(WORK, "checkpoints")
PLOT_DIR  = os.path.join(WORK, "plots")
RESULTS_CSV = os.path.join(WORK, "results_bi_encoder.csv")
for d in (CACHE_DIR, CKPT_DIR, PLOT_DIR):
    os.makedirs(d, exist_ok=True)

# 10 view CV, 5 view JD -> 50 cap view
CV_VIEWS = ['skills', 'education', 'year_experience', 'experience', 'responsibilities',
            'projects', 'publications', 'achievement', 'certification', 'employment_history']
JD_VIEWS = ['skills', 'education', 'year_experience_requirement', 'experience_requirement', 'responsibilities']
NUM_PAIRS = len(CV_VIEWS) * len(JD_VIEWS)  # 50

# 4 encoder se duoc sweep tu dong (Run-All chay het, khong can sua tay)
ENCODERS = [
    "BAAI/bge-base-en-v1.5",
    "thenlper/gte-base",
    "jjzha/jobbert-base-cased",
    "lwolfrum2/careerbert-g",
]

MAX_LEN = 256
EPOCHS  = 150
BATCH   = 32

In [4]:
# =============================== HELPERS ===============================
def process_view(view_data):
    """missing_section -> [EMPTY_SECTION]; tra ve (text, exists_flag)."""
    if view_data is None:
        return "[EMPTY_SECTION]", 0
    if isinstance(view_data, list):
        clean = [str(x) for x in view_data if str(x).lower() != "missing_section"]
        if not clean:
            return "[EMPTY_SECTION]", 0
        return ". ".join(clean), 1
    val = str(view_data).lower()
    if val in ("missing_section", "", "none"):
        return "[EMPTY_SECTION]", 0
    return str(view_data), 1


def load_json(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        return json.load(f)


def read_split_rows(split):
    """Doc match + mismatch CSV cua 1 split -> list (cv, jd, score, bin_label)."""
    rows = []
    for label, b in (("match", 1), ("mismatch", 0)):
        p = os.path.join(SPLIT_ROOT, split, label, f"{split}_{label}.csv")
        df = pd.read_csv(p)
        for _, r in df.iterrows():
            rows.append((r["cv"], r["jd"], float(r["score"]), b))
    return rows


def compute_metrics(y_reg, y_bin, preds):
    preds = np.asarray(preds); y_reg = np.asarray(y_reg); y_bin = np.asarray(y_bin)
    pred_bin = (preds >= 0.5).astype(int)
    return {
        "MAE": mean_absolute_error(y_reg, preds),
        "MSE": mean_squared_error(y_reg, preds),
        "Accuracy": accuracy_score(y_bin, pred_bin),
        "Precision": precision_score(y_bin, pred_bin, zero_division=0),
        "Recall": recall_score(y_bin, pred_bin, zero_division=0),
        "F1-Score": f1_score(y_bin, pred_bin, zero_division=0),
    }


RESULT_COLS = ["Encoding Model", "Fusion", "MAE", "MSE",
               "Accuracy", "Precision", "Recall", "F1-Score"]


def append_result_csv(path, encoder, fusion, m):
    row = {"Encoding Model": encoder, "Fusion": fusion,
           "MAE": round(m["MAE"], 4), "MSE": round(m["MSE"], 4),
           "Accuracy": round(m["Accuracy"], 4), "Precision": round(m["Precision"], 4),
           "Recall": round(m["Recall"], 4), "F1-Score": round(m["F1-Score"], 4)}
    df = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=RESULT_COLS)
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df = df.drop_duplicates(subset=["Encoding Model", "Fusion"], keep="last")
    df = df[RESULT_COLS]
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return df


def plot_history(hist, title, save_path):
    x = np.arange(1, len(hist["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(x, hist["train_loss"], label=f"Train Loss (final={hist['train_loss'][-1]:.4f})")
    ax1.plot(x, hist["val_loss"],   label=f"Val Loss (final={hist['val_loss'][-1]:.4f})")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.set_title(f"{title} - Loss"); ax1.legend(); ax1.grid(True)
    ax2.plot(x, hist["train_acc"], label=f"Train Acc (final={hist['train_acc'][-1]:.4f})")
    ax2.plot(x, hist["val_acc"],   label=f"Val Acc (final={hist['val_acc'][-1]:.4f})")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
    ax2.set_title(f"{title} - Accuracy"); ax2.legend(); ax2.grid(True)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show(); plt.close()

In [5]:
# =============================== LOAD ENCODER ===============================
def load_encoder(name):
    """Tra ve (tokenizer, encoder_frozen, hidden_dim, slug).
    Ca 4 encoder deu la BERT base (bge / gte / jobbert / careerbert) ->
    AutoModel nap BertModel goc va tra ve last_hidden_state -> mean-pool downstream.
    """
    slug = name.split("/")[-1]
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    enc = AutoModel.from_pretrained(name, trust_remote_code=True)
    enc = enc.to(device).eval()
    for p in enc.parameters():
        p.requires_grad_(False)
    hidden = enc.config.hidden_size
    # Smoke-test: fail-fast neu nap nham head (vd jobbert tag MaskedLM) -> khong co last_hidden_state
    with torch.no_grad():
        _t = tok(["test"], ["test"], return_tensors="pt", truncation=True, max_length=8).to(device)
        _o = enc(**_t)
    assert hasattr(_o, "last_hidden_state"), \
        f"{slug}: model khong tra ve last_hidden_state (co the nap nham classification head)"
    print(f"[{slug}] hidden_dim={hidden} | last_hidden_state OK")
    return tok, enc, hidden, slug

In [6]:
# =============== TRICH DAC TRUNG (BI-ENCODER: cosine giua 2 embedding) ===============
def mean_pool(last_hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).float()
    summed = (last_hidden * mask).sum(1)
    counts = mask.sum(1).clamp(min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_texts(texts, tok, enc):
    """Encode RIENG tung doan -> embedding mean-pool, L2-normalize (np.float32)."""
    out = []
    B = 64
    for i in range(0, len(texts), B):
        t = tok(texts[i:i+B], padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors="pt").to(device)
        o = enc(**t)
        emb = mean_pool(o.last_hidden_state, t["attention_mask"])
        emb = F.normalize(emb, dim=-1)
        out.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(out, 0)


def prepare_data(split, tok, enc, slug):
    """Moi mau -> 50 cosine(cv_view_i, jd_view_j) + mask 50 chieu. Cache .npz theo slug+split."""
    cache = os.path.join(CACHE_DIR, f"{slug}_{split}_bi.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        print(f"  cache: {cache} | X={d['X'].shape}")
        return d["X"], d["M"], d["yreg"], d["ybin"]

    rows = read_split_rows(split)
    cv_cache, jd_cache = {}, {}

    def get_cv(fn):
        if fn not in cv_cache:
            obj = load_json(os.path.join(CV_DIR, fn)); j = obj.get("resume_structured", obj)
            texts, ex = zip(*[process_view(j.get(v)) for v in CV_VIEWS])
            cv_cache[fn] = (embed_texts(list(texts), tok, enc), np.array(ex, dtype=np.float32))
        return cv_cache[fn]

    def get_jd(fn):
        if fn not in jd_cache:
            obj = load_json(os.path.join(JD_DIR, fn)); j = obj.get("job_description_structured", obj)
            texts, ex = zip(*[process_view(j.get(v)) for v in JD_VIEWS])
            jd_cache[fn] = (embed_texts(list(texts), tok, enc), np.array(ex, dtype=np.float32))
        return jd_cache[fn]

    X, M, yreg, ybin = [], [], [], []
    for cvfn, jdfn, score, b in tqdm(rows, desc=f"[{slug}] bi {split}"):
        ce, cex = get_cv(cvfn); je, jex = get_jd(jdfn)
        sim = ce @ je.T              # (10,5) = cosine (da normalize)
        mask = np.outer(cex, jex)    # (10,5)
        X.append(sim.flatten()); M.append(mask.flatten())
        yreg.append(score); ybin.append(b)

    X = np.asarray(X, np.float32); M = np.asarray(M, np.float32)
    yreg = np.asarray(yreg, np.float32); ybin = np.asarray(ybin, np.float32)
    np.savez_compressed(cache, X=X, M=M, yreg=yreg, ybin=ybin)
    return X, M, yreg, ybin

In [ ]:
# =============== DATASET + 4 FUSION (input = 50 cosine da loc mask) ===============
class BiDataset(Dataset):
    def __init__(self, X, M, yreg, ybin):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.M = torch.tensor(M, dtype=torch.float32)
        self.yreg = torch.tensor(yreg, dtype=torch.float32).view(-1, 1)
        self.ybin = torch.tensor(ybin, dtype=torch.float32).view(-1, 1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return self.X[i] * self.M[i], self.yreg[i], self.ybin[i]   # Hadamard S*M


class WeightedSumFusion(nn.Module):
    def __init__(self, input_dim=50):
        super().__init__()
        self.layer = nn.Linear(input_dim, 1)
    def forward(self, x): return torch.sigmoid(self.layer(x))


class FusionMLP(nn.Module):
    def __init__(self, input_dim=50):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)


class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.ln = nn.LayerNorm(dim); self.fc = nn.Linear(dim, dim)
    def forward(self, x): return x + F.relu(self.fc(self.ln(x)))


class FusionResidualMLP(nn.Module):
    def __init__(self, input_dim=50):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU())
        self.res1, self.res2 = ResidualBlock(256), ResidualBlock(256)
        self.hidden = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
                                    nn.Linear(128, 64), nn.ReLU())
        self.out = nn.Sequential(nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, x):
        x = self.inp(x); x = self.res1(x); x = self.res2(x); x = self.hidden(x)
        return self.out(x)


class FusionAttention(nn.Module):
    def __init__(self, num_views=50, d_model=64, nhead=8):
        super().__init__()
        self.proj = nn.Linear(1, d_model)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                           dim_feedforward=256, dropout=0.2, batch_first=True)
        self.attn = nn.TransformerEncoder(layer, num_layers=3)
        self.head = nn.Sequential(nn.Linear(d_model * num_views, 128), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(128, 1), nn.Sigmoid())
    def forward(self, x):
        b = x.shape[0]
        h = self.proj(x.unsqueeze(-1))   # (B,50,d)
        h = self.attn(h)
        return self.head(h.reshape(b, -1))


FUSIONS = {
    "weighted_sum": lambda: WeightedSumFusion(NUM_PAIRS),
    "mlp":          lambda: FusionMLP(NUM_PAIRS),
    "residual_mlp": lambda: FusionResidualMLP(NUM_PAIRS),
    "attention":    lambda: FusionAttention(NUM_PAIRS),
}

In [ ]:
# =============== TRAIN + EVAL ===============
def train_one(model, train_loader, val_loader, ckpt):
    model = model.to(device)
    crit_reg = nn.MSELoss(); crit_bin = nn.BCELoss()
    opt = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best = float("inf")
    for ep in range(EPOCHS):
        model.train(); tl = 0.0; tp, tt = [], []
        for x, yr, yb in train_loader:
            x, yr, yb = x.to(device), yr.to(device), yb.to(device)
            opt.zero_grad(); out = model(x)
            loss = crit_reg(out, yr) + 0.5 * crit_bin(out, yb)
            loss.backward(); opt.step(); tl += loss.item()
            tp += list((out.detach().cpu().numpy().flatten() >= 0.5).astype(int))
            tt += list(yb.cpu().numpy().flatten().astype(int))
        model.eval(); vl = 0.0; vp, vt = [], []
        with torch.no_grad():
            for x, yr, yb in val_loader:
                x, yr, yb = x.to(device), yr.to(device), yb.to(device)
                out = model(x); vl += (crit_reg(out, yr) + 0.5 * crit_bin(out, yb)).item()
                vp += list((out.cpu().numpy().flatten() >= 0.5).astype(int))
                vt += list(yb.cpu().numpy().flatten().astype(int))
        av = vl / len(val_loader); sched.step(av)
        hist["train_loss"].append(tl / len(train_loader)); hist["val_loss"].append(av)
        hist["train_acc"].append(accuracy_score(tt, tp)); hist["val_acc"].append(accuracy_score(vt, vp))
        if av < best:
            best = av; torch.save(model.state_dict(), ckpt)
    return hist


def evaluate(model, test_loader, ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=device)); model.eval()
    preds, yr, yb = [], [], []
    with torch.no_grad():
        for x, r, b in test_loader:
            out = model(x.to(device))
            preds += list(out.cpu().numpy().flatten())
            yr += list(r.numpy().flatten()); yb += list(b.numpy().flatten())
    return compute_metrics(yr, yb, preds)

In [ ]:
# =============== DRIVER: Run-All tu dong sweep 4 ENCODER x 4 FUSION ===============
# Moi encoder bao boc try/except: 1 model loi se bi log + bo qua, KHONG chet ca run.
for ENCODER_NAME in ENCODERS:
    print("\n" + "=" * 80)
    print("ENCODER:", ENCODER_NAME)
    print("=" * 80)
    try:
        tok, enc, hidden, slug = load_encoder(ENCODER_NAME)

        Xtr, Mtr, ytr, btr = prepare_data("train", tok, enc, slug)
        Xv,  Mv,  yv,  bv  = prepare_data("val",   tok, enc, slug)
        Xte, Mte, yte, bte = prepare_data("test",  tok, enc, slug)

        # Feature da cache xong -> giai phong encoder (chi train fusion head)
        del enc, tok; gc.collect(); torch.cuda.empty_cache()

        train_loader = DataLoader(BiDataset(Xtr, Mtr, ytr, btr), batch_size=BATCH, shuffle=True)
        val_loader   = DataLoader(BiDataset(Xv,  Mv,  yv,  bv),  batch_size=BATCH)
        test_loader  = DataLoader(BiDataset(Xte, Mte, yte, bte), batch_size=BATCH)

        for FUSION_NAME, build in FUSIONS.items():
            ckpt = os.path.join(CKPT_DIR, f"{FUSION_NAME}_{slug}.pth")
            model = build()
            hist = train_one(model, train_loader, val_loader, ckpt)
            plot_history(hist, f"{FUSION_NAME} | {slug}",
                         os.path.join(PLOT_DIR, f"{FUSION_NAME}-{slug}.png"))
            m = evaluate(model, test_loader, ckpt)
            df = append_result_csv(RESULTS_CSV, ENCODER_NAME, FUSION_NAME, m)
            print(f"  [{slug} | {FUSION_NAME}] "
                  f"MAE={m['MAE']:.4f} MSE={m['MSE']:.4f} Acc={m['Accuracy']:.4f} F1={m['F1-Score']:.4f}")
    except Exception as e:
        import traceback
        print(f"!! BO QUA encoder {ENCODER_NAME} do loi: {e}")
        traceback.print_exc()
        gc.collect(); torch.cuda.empty_cache()
        continue

print("\nXONG. Bang ket qua:")

In [ ]:
# =============== BANG KET QUA TONG HOP ===============
res = pd.read_csv(RESULTS_CSV)
print("Luu tai:", RESULTS_CSV)
res